# Team 3 — Face Detection + Stable Tile Tracking → JSON for Team 4

**Presentation requirement (Step 4):** video in → faces + bounding boxes + total participants.
**Team 4's format:** JSON schema below, normalized 0–1, with **stable** `face_id`s.

Reads & writes the shared Drive folder `gong` (no uploads). Produces:
1. `faces_output.json` — deliverable for Team 4.
2. `annotated_output.mp4` — boxes + `face_id` drawn on, for your visual check.

**How identity is kept stable:** YOLOv8-face detects the faces; because a Zoom Gallery has
fixed tiles, each participant's identity is locked to the **position of their face**, so a person
keeps the same `face_id` for the whole video. **Two people sharing one tile each get their own
`face_id`** (this covers the "several people on one screen / in one room" case), while a face that
only appears briefly is treated as background and dropped.

> Runtime → Change runtime type → **GPU**, then run top to bottom.

## 1. Install

In [2]:
!pip -q install ultralytics huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.0 MB/s eta 0:00:00


## 2. Connect Google Drive
Approve access. Confirms it can see `zoom_meeting.mp4`.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/gong"
print("Files in folder:", os.listdir(PROJECT_DIR))

Mounted at /content/drive
Files in folder: ['zoom_meeting.mp4', 'face_detection_team3.ipynb']


## 3. Run — detection + stable tile identity → JSON + annotated video
Quick test first: set `MAX_FRAMES = 900` (~30s). Set back to `None` for the full run.

Two knobs if needed:
- `SLOT_RADIUS` — how close two detections must be to count as the same tile (raise if one
  person gets split into two ids; lower if two tiles merge).
- `MIN_PRESENCE` — a real participant must appear in at least this fraction of frames
  (raises/lowers how aggressively background/partial faces are filtered).

In [7]:
import cv2, json, os, math
from ultralytics import YOLO
from huggingface_hub import hf_hub_download
import itertools as _it

# ---------- CONFIG ----------
PROJECT_DIR  = "/content/drive/MyDrive/gong"
VIDEO_PATH   = os.path.join(PROJECT_DIR, "zoom_meeting.mp4")
OUTPUT_JSON  = os.path.join(PROJECT_DIR, "faces_output.json")
ANNOTATED    = os.path.join(PROJECT_DIR, "annotated_raw.mp4")

CONF         = 0.30
IMGSZ        = 1280
TRACKER      = "bytetrack.yaml"
MAX_FRAMES   = None

SLOT_RADIUS  = 0.16
MIN_PRESENCE = 0.08
# ----------------------------

model_path = hf_hub_download(
    repo_id="arnabdhar/YOLOv8-Face-Detection",
    filename="model.pt"
)
model = YOLO(model_path)

probe = cv2.VideoCapture(VIDEO_PATH)
fps   = probe.get(cv2.CAP_PROP_FPS) or 30.0
W     = int(probe.get(cv2.CAP_PROP_FRAME_WIDTH))
H     = int(probe.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(probe.get(cv2.CAP_PROP_FRAME_COUNT))
probe.release()

print(f"{W}x{H} @ {fps:.2f}fps, ~{total} frames")

# ---- PASS A ----
per_frame = []

results = model.track(
    source=VIDEO_PATH,
    stream=True,
    persist=True,
    tracker=TRACKER,
    conf=CONF,
    imgsz=IMGSZ,
    verbose=False
)

for fi, res in enumerate(results):
    if MAX_FRAMES is not None and fi >= MAX_FRAMES:
        break

    dets = []

    if res.boxes is not None and len(res.boxes):
        for (x1, y1, x2, y2) in res.boxes.xyxy.cpu().numpy():
            nx1, ny1, nx2, ny2 = x1 / W, y1 / H, x2 / W, y2 / H
            cx = (nx1 + nx2) / 2
            cy = (ny1 + ny2) / 2
            dets.append((cx, cy, nx1, ny1, nx2, ny2))

    per_frame.append(dets)

    if fi % 500 == 0:
        print(f"  detect frame {fi}/{total}")

n_frames = len(per_frame)
print(f"detected frames: {n_frames}")

# ---- FIXED SLOT CLUSTERING ----
slots = []

for dets in per_frame:
    for (cx, cy, *_) in dets:

        best, bestd = None, 1e9

        for s in slots:
            d = math.hypot(cx - s["cx"], cy - s["cy"])
            if d < bestd:
                best, bestd = s, d

        if best is not None and bestd <= SLOT_RADIUS:
            n = best["count"]
            best["cx"] = (best["cx"] * n + cx) / (n + 1)
            best["cy"] = (best["cy"] * n + cy) / (n + 1)
            best["count"] += 1
        else:
            slots.append({
                "cx": cx,
                "cy": cy,
                "count": 1
            })

real = [
    s for s in slots
    if s["count"] >= MIN_PRESENCE * n_frames
]

real.sort(key=lambda s: (round(s["cy"], 1), s["cx"]))

for i, s in enumerate(real):
    s["label"] = f"face_{i + 1:02d}"

print(f"\nraw slots: {len(slots)}  ->  real participants: {len(real)}")

for s in real:
    pct = 100 * s["count"] / n_frames
    print(
        f'{s["label"]}: center=({s["cx"]:.2f},{s["cy"]:.2f}) '
        f'present {pct:.0f}% of frames'
    )

# ---- PASS B ----
cap = cv2.VideoCapture(VIDEO_PATH)
writer = cv2.VideoWriter(
    ANNOTATED,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (W, H)
)

frames_out = []

def assign(cx, cy):
    best, bestd = None, 1e9

    for s in real:
        d = math.hypot(cx - s["cx"], cy - s["cy"])
        if d < bestd:
            best, bestd = s, d

    if best is not None and bestd <= SLOT_RADIUS:
        return best["label"]

    return None

for fi in range(n_frames):
    ok, frame = cap.read()
    if not ok:
        break

    chosen = {}

    for (cx, cy, nx1, ny1, nx2, ny2) in per_frame[fi]:
        lab = assign(cx, cy)
        if lab is None:
            continue

        s = next(s for s in real if s["label"] == lab)
        d = math.hypot(cx - s["cx"], cy - s["cy"])

        if lab not in chosen or d < chosen[lab][1]:
            chosen[lab] = ((nx1, ny1, nx2, ny2), d)

    faces = []

    for lab, ((nx1, ny1, nx2, ny2), _) in sorted(chosen.items()):
        faces.append({
            "face_id": lab,
            "bounding_box": {
                "x_min": round(float(nx1), 4),
                "y_min": round(float(ny1), 4),
                "width": round(float(nx2 - nx1), 4),
                "height": round(float(ny2 - ny1), 4)
            }
        })

        x1, y1, x2, y2 = int(nx1 * W), int(ny1 * H), int(nx2 * W), int(ny2 * H)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, lab, (x1, max(y1 - 8, 14)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    writer.write(frame)

    frames_out.append({
        "frame_index": fi,
        "timestamp_ms": int(round(fi / fps * 1000)),
        "faces_detected": faces
    })

    if fi % 500 == 0:
        print(f"  render frame {fi}/{n_frames}")

cap.release()
writer.release()

output = {
    "total_participants_detected": int(len(real)),
    "fps": float(round(fps, 2)),
    "frames": frames_out
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(output, f, indent=2, default=float)

print(f"\nDONE -> saved to {PROJECT_DIR}")
print(f"participants: {len(real)} | frames: {len(frames_out)}")

506x848 @ 29.26fps, ~22008 frames
  detect frame 0/22008
  detect frame 500/22008
  detect frame 1000/22008
  detect frame 1500/22008
  detect frame 2000/22008
  detect frame 2500/22008
  detect frame 3000/22008
  detect frame 3500/22008
  detect frame 4000/22008
  detect frame 4500/22008
  detect frame 5000/22008
  detect frame 5500/22008
  detect frame 6000/22008
  detect frame 6500/22008
  detect frame 7000/22008
  detect frame 7500/22008
  detect frame 8000/22008
  detect frame 8500/22008
  detect frame 9000/22008
  detect frame 9500/22008
  detect frame 10000/22008
  detect frame 10500/22008
  detect frame 11000/22008
  detect frame 11500/22008
  detect frame 12000/22008
  detect frame 12500/22008
  detect frame 13000/22008
  detect frame 13500/22008
  detect frame 14000/22008
  detect frame 14500/22008
  detect frame 15000/22008
  detect frame 15500/22008
  detect frame 16000/22008
  detect frame 16500/22008
  detect frame 17000/22008
  detect frame 17500/22008
  detect frame 180

## 4. Watch the annotated result
Re-encodes to H.264 (saved to the folder) and plays inline. Check that **each tile keeps the same
`face_id` from start to end**. If it's too long to play inline, open it from Drive.

In [8]:
import os, subprocess
ANNOT_H264 = os.path.join(PROJECT_DIR, "annotated_output.mp4")
subprocess.run(["ffmpeg", "-y", "-loglevel", "error",
                "-i", ANNOTATED, "-vcodec", "libx264", "-pix_fmt", "yuv420p",
                ANNOT_H264], check=True)
size_mb = os.path.getsize(ANNOT_H264) / 1e6
print(f"annotated_output.mp4 = {size_mb:.1f} MB (saved in {PROJECT_DIR})")
if size_mb <= 40:
    from IPython.display import HTML
    from base64 import b64encode
    b64 = b64encode(open(ANNOT_H264, "rb").read()).decode()
    display(HTML(f'<video width=400 controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
else:
    print("Too large to embed inline — open it from the gong folder in Drive.")

annotated_output.mp4 = 64.1 MB (saved in /content/drive/MyDrive/gong)
Too large to embed inline — open it from the gong folder in Drive.


## 5. Sanity check the JSON

In [9]:
import json
data = json.load(open(OUTPUT_JSON))
assert set(data) >= {"total_participants_detected", "fps", "frames"}
bad = sum(1 for fr in data["frames"] for f in fr["faces_detected"]
          for k in ("x_min", "y_min", "width", "height")
          if not (0.0 <= f["bounding_box"][k] <= 1.0))
print("participants:", data["total_participants_detected"], "(expect 7 if the shared-tile girl is counted)")
print("frames:", len(data["frames"]))
print("out-of-range box values:", bad, "(must be 0)")
print(json.dumps(next(fr for fr in data["frames"] if fr["faces_detected"]), indent=2))

participants: 7 (expect 7 if the shared-tile girl is counted)
frames: 22008
out-of-range box values: 0 (must be 0)
{
  "frame_index": 0,
  "timestamp_ms": 0,
  "faces_detected": [
    {
      "face_id": "face_01",
      "bounding_box": {
        "x_min": 0.7799,
        "y_min": 0.0984,
        "width": 0.115,
        "height": 0.0966
      }
    },
    {
      "face_id": "face_02",
      "bounding_box": {
        "x_min": 0.1032,
        "y_min": 0.148,
        "width": 0.2519,
        "height": 0.1407
      }
    },
    {
      "face_id": "face_04",
      "bounding_box": {
        "x_min": 0.2349,
        "y_min": 0.4023,
        "width": 0.1255,
        "height": 0.1007
      }
    },
    {
      "face_id": "face_05",
      "bounding_box": {
        "x_min": 0.8126,
        "y_min": 0.4062,
        "width": 0.1591,
        "height": 0.1162
      }
    },
    {
      "face_id": "face_06",
      "bounding_box": {
        "x_min": 0.4287,
        "y_min": 0.6635,
        "width": 0.199

## Done
`faces_output.json` + `annotated_output.mp4` are in the shared **gong** folder.
Tell Team 4 the JSON is ready — they read it (and the video) straight from Drive.